<a href="https://colab.research.google.com/github/Muskankumari13/Week1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The FlyRank research paper reports that AI-assisted SEO can improve content performance.

### My methodology question

How was "content performance" measured? Was it based on clicks, impressions, rankings, conversions, or another metric? Knowing the exact label helps judge whether the conclusion is supported by the data.

---

## Finding 2

The paper suggests that combining AI with human review produces better outcomes.

### My methodology question

How was "better" measured? Was quality evaluated by human reviewers, user engagement, or search engine rankings? Also, was the same validation process applied to every comparison group?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Validation

Previous random train-test split accuracy: **0.668**

TimeSeriesSplit accuracy: **0.66064**

The TimeSeriesSplit score is slightly lower than the random split score. This suggests the random split may have given a slightly optimistic estimate. Using a time-aware validation provides a more realistic estimate of how the model may perform on future data.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Create target
df["target"] = (df["trend_direction"] == "down").astype(int)

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "search_volume",
    "content_age_days",
    "clicks_90d",
    "sessions_90d"
]

X = df[features].fillna(0)
y = df["target"]

# Week 5 random split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Previous Random Split Accuracy:", accuracy)

Previous Random Split Accuracy: 0.668


In [2]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring="accuracy"
)

print("TimeSeriesSplit Accuracy:", scores.mean())
print("Fold Scores:", scores)

TimeSeriesSplit Accuracy: 0.66064
Fold Scores: [0.6524 0.6558 0.6664 0.6666 0.662 ]


The time-aware validation produced a more realistic estimate than the random train-test split because it evaluates the model across sequential data splits.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

I checked my feature set for possible leakage.

- No target column was used as an input feature.
- No future information was included.
- Data preprocessing was performed only on the training data.
- Duplicate records were checked before training.

Based on this review, I did not find obvious evidence of target leakage.

In [3]:
# Leakage audit checks
print("Target column used as feature:", "target" in X.columns)
print("Missing values:", X.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Target column used as feature: False
Missing values: 0
Duplicate rows: 0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Revised Claim

On this dataset, the Random Forest model achieved an observed accuracy of **0.668** using a random train-test split and **0.66064** using TimeSeriesSplit. These results provide directional evidence and may be useful for decision-support, but they should not be interpreted as proof that the model will perform the same on all future datasets.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.